# Symbolic regression on micrometeorological data (Rebio Jaru)

Notebook version of the pipeline. Run the cells in order (`Shift+Enter`).

1. Setup and imports
2. Synthetic baseline dataset (and, optionally, the real spreadsheet)
3. Quick PySR tests (an algebraic law and the Jaru temperature ODE)
4. Full PySR benchmark, saved next to the cloud results
5. Comparison of all methods (PySR, Operon, SINDy, gplearn, AI Feynman)

The first `import pysr` installs Julia automatically (once, several minutes).

In [ ]:
# 1. Setup: run once per environment (it is quick when everything is already installed)
%pip install -q -r requirements.txt

In [ ]:
import sys, time, csv
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()                      # folder that contains this notebook
SRC, BENCH = ROOT / "src", ROOT / "src" / "sr_benchmark"
for p in (SRC, BENCH):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from data_inspection import GeneratorConfig, generate_synthetic, inspect_dataframe, diurnal_composite, save_dataset
from tasks import STATIC_TASKS, JARU_T, LORENZ, sample_static, add_noise, simulate_jaru
from derivatives import savitzky_golay
from methods import METHODS, complexity
import run_benchmark as rb
from summarize import load
pd.set_option("display.max_colwidth", 120)
print("OK, project root:", ROOT)

## 2. Synthetic baseline dataset

In [ ]:
cfg = GeneratorConfig(seed=42)
cfg.site.n_days = 365
df = generate_synthetic(cfg)
path = save_dataset(df, cfg, ROOT / "data" / "synthetic")
print("Saved:", path)
inspect_dataframe(df[["R_liq", "T_ar_62m", "UR_62m", "VV_ane_62m", "P_atm", "PRP", "dTdt_true_K_per_h"]])

In [ ]:
week = df.iloc[24 * 2 * 200 : 24 * 2 * 207]   # one week in the dry season
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(week.timestamp, week.R_liq); ax[0].set_ylabel("Rn (W m$^{-2}$)")
ax[1].plot(week.timestamp, week.T_ar_62m); ax[1].set_ylabel("T (°C)")
ax[2].plot(week.timestamp, week.UR_62m); ax[2].set_ylabel("RH (%)")
plt.tight_layout(); plt.show()

In [ ]:
# Optional: structural summary of the real spreadsheet (place the .xlsx next to this notebook)
xlsx = ROOT / "Dados Rebio Jaru 2006 a 2008.xlsx"
if xlsx.exists():
    real = pd.read_excel(xlsx, sheet_name="DADOS OK")
    print(real.shape)
    display(inspect_dataframe(real).head(15))
else:
    print("Spreadsheet not found; skipping.")

## 3. Quick PySR tests

The first cell installs/loads Julia. Each fit has a hard time limit (`budget`, seconds).

In [ ]:
t0 = time.perf_counter()
from pysr import PySRRegressor
print(f"PySR ready in {time.perf_counter() - t0:.0f} s")

In [ ]:
def r2(y, p):
    return 1 - np.mean((y - p) ** 2) / np.var(y)

def try_static(task_name, method="pysr", noise=0.0, budget=45, seed=0):
    task = next(t for t in STATIC_TASKS if t.name == task_name)
    rng = np.random.default_rng(seed)
    X, y = sample_static(task, 1000, rng)
    Xt, yt = sample_static(task, 2000, rng, test=True)
    y = add_noise(y, noise, np.random.default_rng(seed + 1))
    fit = METHODS[method](X, y, task.variables, budget, seed)
    print(f"{task_name} | {method} | noise={noise} | {fit.seconds:.0f} s")
    print("  truth    :", task.formula)
    print("  found    :", fit.expression)
    print(f"  R2 extrap: {r2(yt, fit.predict(Xt)):.5f}   complexity: {complexity(fit.expression)}")
    return fit

_ = try_static("stefan_boltzmann", "pysr", noise=0.0, budget=30)

In [ ]:
# Same law with 5 % noise, and a harder one (bulk sensible heat flux)
_ = try_static("stefan_boltzmann", "pysr", noise=0.05, budget=30)
_ = try_static("bulk_sensible_heat", "pysr", noise=0.0, budget=45)

In [ ]:
# Jaru temperature ODE: dT/dt = 0.0055*Rn - 0.30*T + 6.9, from a noisy 60-day trajectory
sim = simulate_jaru(n_days=60, seed=0)
rng = np.random.default_rng(2)
T_noisy = sim["states"] + rng.normal(0, 0.05 * sim["states"].std(), sim["states"].shape)
T_s, dTdt = savitzky_golay(sim["t"], T_noisy, window=7, polyorder=2)   # smooth derivative, NOT np.diff
X = np.column_stack([T_s[:, 0], sim["forcing"][:, 0]])
fit = METHODS["pysr"](X[10:-10], dTdt[10:-10, 0], ["T", "Rn"], 45, 0)
print("truth: 0.0055*Rn - 0.30*T + 6.9")
print("found:", fit.expression, f"({fit.seconds:.0f} s)")

## 4. Full PySR benchmark

Same tasks, noise levels and scoring as the cloud run (about 1 h). Results are written to
`src/sr_benchmark/results/static_pysr.csv` and `dynamic_pysr.csv`. Set `QUICK = True` to test with 2 tasks.

In [ ]:
QUICK = False
BUDGET = 45   # seconds per static task; dynamic targets use 30 s

def run_suite(suite, methods, noises, budget, out, only=None):
    out = BENCH / "results" / out
    out.unlink(missing_ok=True)
    with out.open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=rb.FIELDS); w.writeheader()
        def writer(row):
            w.writerow({k: row.get(k, "") for k in rb.FIELDS}); fh.flush()
            print(f"{row['task']:<24} {row.get('target',''):<6} noise={row['noise']:<5} "
                  f"{row.get('estimator','-'):<18} {row['method']:<8} R2={row.get('r2_extrap', float('nan')):.4f}")
        if suite == "static":
            rb.static_suite(methods, noises, budget, 0, 1000, writer, only)
        else:
            rb.dynamic_suite(methods, noises, budget, 0, writer)
    return out

if QUICK:
    run_suite("static", ["pysr"], [0.0], 30, "quick_pysr.csv", only=["stefan_boltzmann", "bulk_sensible_heat"])
else:
    run_suite("static", ["pysr"], [0.0, 0.05], BUDGET, "static_pysr.csv")
    run_suite("dynamic", ["pysr"], [0.0, 0.01, 0.05], 30, "dynamic_pysr.csv")

## 5. Comparison of all methods

In [ ]:
csvs = [p for p in (BENCH / "results").glob("*.csv") if not p.name.startswith("quick")]
res = load([str(p) for p in csvs])
static = res[res.suite == "static"]
summary = static.groupby(["method", "noise"]).agg(
    tasks=("task", "nunique"),
    accuracy_solutions=("accuracy_solution", "sum"),
    parsimonious=("parsimonious", "sum"),
    median_r2=("r2_extrap", "median"),
    median_complexity=("complexity", "median"),
    median_seconds=("seconds", "median"),
).round(4)
summary

In [ ]:
static.pivot_table(index="task", columns=["method", "noise"], values="r2_extrap").clip(lower=-1).round(4)

In [ ]:
dyn = res[res.suite == "dynamic"]
dyn.pivot_table(index=["method", "estimator"], columns="noise", values="parsimonious", aggfunc="sum")

In [ ]:
# Accuracy vs. parsimonious solutions, static suite, noise-free
s0 = summary.xs(0.0, level="noise").sort_values("accuracy_solutions")
ax = s0[["accuracy_solutions", "parsimonious"]].plot.barh(figsize=(8, 4))
ax.set_xlabel("solutions (out of 10 laws)"); ax.set_title("Static suite, noise-free")
plt.tight_layout(); plt.show()

In [ ]:
# Expressions found (pick a task)
TASK = "bulk_sensible_heat"
res[(res.task == TASK)][["method", "noise", "estimator", "r2_extrap", "complexity", "expression"]]

## 6. New equations from the real Jaru record vs. existing equations

Each classical law keeps its form but its constants are re-fitted on 2006-2007; every
discovery engine sees the same inputs; everything is scored on 2008 (out-of-time).
Place `Dados Rebio Jaru 2006 a 2008.xlsx` next to this notebook.

Targets: `net_radiation` (Rn vs PAR), `relative_humidity` (RH vs T and daily Tmin),
`temperature_tendency` (dT/dt vs Rn, T, wind). Add your own in `src/real_discovery.py` (`TARGETS`).

In [ ]:
import real_discovery as rd
jaru = rd.load_jaru(ROOT / "Dados Rebio Jaru 2006 a 2008.xlsx")
print(jaru.shape)
jaru.groupby("year")[["Rn", "PAR", "RH", "T22", "WS", "dTdt"]].apply(lambda d: d.notna().mean()).round(2)

In [ ]:
SEASON = None          # None, "wet" or "dry"
METHODS_REAL = ["pysr", "operon", "sindy_poly2"]
cols = ["model", "rmse_test", "r2_test", "constants", "complexity", "bic_test", "check", "expression"]
real_results = {}
for name in rd.TARGETS:
    res = rd.compare(jaru, name, METHODS_REAL, budget=90, season=SEASON)
    real_results[name] = res
    print(f"\n=== {name}: train n={res.attrs['n_train']}, test (2008) n={res.attrs['n_test']}")
    display(res[[c for c in cols if c in res]])
pd.concat(real_results.values()).to_csv(BENCH / "results" / "real_discovery_local.csv", index=False)

In [ ]:
# PySR Pareto front for one target: every candidate equation, with its error on 2008,
# against the re-fitted existing equation (dashed line)
TARGET = "temperature_tendency"
front = rd.pysr_pareto(jaru, TARGET, budget=180, season=SEASON)
base = real_results[TARGET].iloc[0]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(front.complexity, front.rmse_test, "o-", label="PySR Pareto front")
ax.axhline(base.rmse_test, ls="--", color="k", label="existing equation (re-fitted)")
ax.set_xlabel("complexity"); ax.set_ylabel("RMSE on 2008"); ax.set_title(TARGET); ax.legend()
plt.tight_layout(); plt.show()
front